# Sinh reframing cho từng bài báo bằng OpenAI API
Sinh 4 tập: emotionally_triggering, neutral, objective, sensational theo prompt bài báo.
- Đọc train_gop.csv
- Gọi OpenAI API với prompt tương ứng
- Lưu kết quả ra 4 file .pkl tương ứng

In [ ]:
import pandas as pd
import openai
import pickle
from tqdm import tqdm
import concurrent.futures

# Thay bằng API key của bạn
client = openai.OpenAI(api_key='YOUR_API_KEY')

df = pd.read_csv('train.csv')
print("SỐ MẪU:", len(df))
articles = df['news'].tolist()

tones = ['emotionally_triggering', 'neutral', 'objective', 'sensational']
reframed = {tone: [] for tone in tones}

def generate_rewrite(article, tone):
    prompt = f"""Rewrite the following Vietnamese article in a {tone} tone (ONLY WRITE TO VIETNAMESE LANGUAGE):
    
    {article}
    """
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=1024,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("Error:", e)
        return ""

for tone in tones:
    print(f"Generating for tone: {tone}")
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        # executor.map trả về generator, dùng tqdm để track
        for rewritten in tqdm(executor.map(lambda art: generate_rewrite(art, tone), articles), total=len(articles)):
            results.append(rewritten)
    
    reframed[tone] = results
    
    # Lưu từng tone ra file pkl
    with open(f'reintel_train_{tone}.pkl', 'wb') as f:
        pickle.dump({'rewritten': reframed[tone]}, f)
    
    print(f'Saved train_{tone}.pkl')
3

KeyboardInterrupt: 

In [ ]:
# Sinh veracity attributions cho từng bài báo (đúng chuẩn SheepDog: từng cặp tone riêng biệt)
import pickle
import openai
from tqdm import tqdm
import concurrent.futures

ATTRS = [
    'Lack of credible sources',
    'False or misleading information',
    'Biased opinion',
    'Inconsistencies with reputable sources'
]

PROMPT = '''Article: {article}\nQuestion: Which of the following problems does this Vietnamese article have?\n- Lack of credible sources\n- False or misleading information\n- Biased opinion\n- Inconsistencies with reputable sources\nIf multiple options apply, provide a comma-separated list ordered from most to least related. Answer "No problems" if none of the options apply.'''

def parse_attribution(ans):
    ans = ans.strip().lower()
    if 'no problems' in ans:
        return [0, 0, 0, 0]
    vec = [0, 0, 0, 0]
    for i, att in enumerate(ATTRS):
        if att.lower() in ans:
            vec[i] = 1
    return vec

client = openai.OpenAI(api_key='YOUR_API_KEY')
def get_veracity_attr(article):
    prompt = PROMPT.format(article=article)
    try:
        response = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=180,
            temperature=0.3
        )
        ans = response.choices[0].message.content.strip()
        return parse_attribution(ans)
    except Exception as e:
        print('Error:', e)
        return [0, 0, 0, 0]

# Đọc các tập reframed (đã có 4 file pkl cho từng tone)
tones = ['emotionally_triggering', 'neutral', 'objective', 'sensational']
reframed = {}
for tone in tones:
    with open(f'reintel_train_{tone}.pkl', 'rb') as f:
        reframed[tone] = pickle.load(f)['rewritten']

# Đọc bản gốc
import pandas as pd
df = pd.read_csv('train.csv')
orig_articles = df['news'].tolist()

# Hàm sinh và lưu attribution cho từng cặp tone
# Cặp 1: objective (mainstream) & emotionally_triggering (tabloid)
print('Generating veracity attribution for objective (mainstream) & emotionally_triggering (tabloid)...')
mainstream_fg = []
tabloid_fg = []
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    for vec in tqdm(executor.map(get_veracity_attr, reframed['objective']), total=len(orig_articles)):
        mainstream_fg.append(vec)
    for vec in tqdm(executor.map(get_veracity_attr, reframed['emotionally_triggering']), total=len(orig_articles)):
        tabloid_fg.append(vec)
with open('fake_standards_objective_emotionally_triggering.pkl', 'wb') as f:
    pickle.dump({'mainstream_fg': mainstream_fg, 'tabloid_fg': tabloid_fg}, f)
print('Saved fake_standards_objective_emotionally_triggering.pkl')

# Cặp 2: neutral (mainstream) & sensational (tabloid)
print('Generating veracity attribution for neutral (mainstream) & sensational (tabloid)...')
mainstream_fg = []
tabloid_fg = []
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    for vec in tqdm(executor.map(get_veracity_attr, reframed['neutral']), total=len(orig_articles)):
        mainstream_fg.append(vec)
    for vec in tqdm(executor.map(get_veracity_attr, reframed['sensational']), total=len(orig_articles)):
        tabloid_fg.append(vec)
with open('fake_standards_neutral_sensational.pkl', 'wb') as f:
    pickle.dump({'mainstream_fg': mainstream_fg, 'tabloid_fg': tabloid_fg}, f)
print('Saved fake_standards_neutral_sensational.pkl')

Generating veracity attribution for objective (mainstream) & emotionally_triggering (tabloid)...


 11%|█         | 979/9227 [09:29<5:03:24,  2.21s/it] 

Error: Error code: 403


  6%|▌         | 549/9227 [04:22<1:09:10,  2.09it/s]



KeyboardInterrupt: 